# 02 — เตรียม Dataset\n\nแบ่งภาพจาก `raw/` เป็น train/val/test โดยแบ่งตาม **sample_id** (ไม่ใช่ random)\nเพื่อป้องกัน data leak — ภาพทุกรูปของหัวเดียวกันจะอยู่ใน split เดียวกันเสมอ\n\n| split | cos | gok | สัดส่วน |\n|---|---|---|---|\n| train | 01–07 | 01–07 | 70% |\n| val   | 08    | 08    | 15% |\n| test  | 09–10 | 09–10 | 15% |

## 1. กำหนดการแบ่ง split

In [ ]:
import os
import shutil

RAW_DIR  = 'raw'
DATA_DIR = 'data'

# แบ่งตาม sample_id — ไม่ใช่ random เพื่อป้องกัน data leak
# IDs 01-10 มีข้อมูล D0-D8 (rich), IDs 11-30 มีแค่ D0-D6 (lean)
# กระจาย rich/lean ให้สมดุลทุก split
SPLIT_MAP = {
    'train': [f'{i:02d}' for i in range(1, 8)] +   # 01–07 (rich)
             [f'{i:02d}' for i in range(11, 25)],  # 11–24 (lean)
    'val':   [f'{i:02d}' for i in range(8, 10)] +  # 08–09 (rich)
             [f'{i:02d}' for i in range(25, 27)],  # 25–26 (lean)
    'test':  ['10'] +                               # 10    (rich)
             [f'{i:02d}' for i in range(27, 31)],  # 27–30 (lean)
}

# แสดงสรุปการแบ่ง
for split, ids in SPLIT_MAP.items():
    print(f'{split:5s} ({len(ids)} หัว): {ids}')

## 2. Copy ภาพไปยัง data/

In [ ]:
import re

# สร้างโฟลเดอร์ปลายทางทั้งหมดก่อน เช่น data/train/cos, data/val/gok, ...
for split in SPLIT_MAP:
    for class_name in ['cos', 'gok']:
        os.makedirs(os.path.join(DATA_DIR, split, class_name), exist_ok=True)

# นับไฟล์ที่ copy แต่ละ split
counts = {'train': 0, 'val': 0, 'test': 0}

for class_name in ['cos', 'gok']:
    src_dir = os.path.join(RAW_DIR, class_name)
    prefix  = 'COS' if class_name == 'cos' else 'GOK'

    for fname in os.listdir(src_dir):
        if not fname.lower().endswith('.jpg'):
            continue

        # ดึงเลข id จากชื่อไฟล์ เช่น COS07_A_D1_E_top.jpg → 07
        m = re.match(rf'^{prefix}(\d+)_', fname, re.IGNORECASE)
        if not m:
            continue
        file_id = m.group(1)  # เช่น '07'

        # หาว่า id นี้อยู่ split ไหน
        target_split = None
        for split, ids in SPLIT_MAP.items():
            if file_id in ids:
                target_split = split
                break

        if target_split is None:
            # id เกิน 10 ข้ามไป (ยังไม่ได้ถ่าย)
            continue

        # copy ไฟล์ไปยังโฟลเดอร์ปลายทาง
        src  = os.path.join(src_dir, fname)
        dst  = os.path.join(DATA_DIR, target_split, class_name, fname)
        shutil.copy2(src, dst)  # copy2 คือ copy พร้อม metadata เช่น วันที่ไฟล์
        counts[target_split] += 1

print('copy เสร็จแล้ว:')
for split, n in counts.items():
    print(f'  {split:5s}: {n} ภาพ')

## 3. ตรวจสอบผลลัพธ์

In [ ]:
# นับภาพในแต่ละโฟลเดอร์ปลายทาง เพื่อยืนยันว่า copy ถูกต้อง
print('=== จำนวนภาพในแต่ละ split/class ===\n')
total = 0
for split in ['train', 'val', 'test']:
    for class_name in ['cos', 'gok']:
        folder = os.path.join(DATA_DIR, split, class_name)
        n = len([f for f in os.listdir(folder) if f.lower().endswith('.jpg')])
        total += n
        print(f'  data/{split}/{class_name}: {n} ภาพ')
    print()

print(f'รวมทั้งหมด: {total} ภาพ')